In [14]:
import os
import requests
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from datetime import datetime, timedelta
import json
from ipynb.fs.full.mongo_loader import rename_and_remove_fields


# odds api key
api_key = os.getenv('ODDS_API_KEY')
root = 'https://api.the-odds-api.com/v4/'

# Mongo 
uri = os.getenv('SBS_V1_MONGO_URI')

# Create a new client and connect to the server
client = MongoClient(uri, server_api=ServerApi('1'))
db = client['SBSV1']
nba_games_historical_collection = db['nba_games_historical']
nba_odds_historical_collection = db['nba_odds_historical']

In [15]:
################ markets reference ######################
#########################################################

nba_team_market = [
    'h2h',
    'spreads',
    'totals'
]

nba_team_market_str = ','.join(nba_team_market)

nba_player_markets = [
    'player_points',
    'player_rebounds',
    'player_assists',
    'player_threes',
    'player_blocks',
    'player_steals',
    'player_blocks_steals',
    'player_turnovers',
    'player_points_rebounds_assists',
    'player_points_rebounds',
    'player_points_assists',
    'player_rebounds_assists',
    'player_first_basket',
    'player_double_double',
    'player_triple_double'
]

nba_player_markets_str = ','.join(nba_player_markets)

basic_bookmakers = [
    'draftkings',
    'fanduel',
    'betmgm'
]
basic_bookmakers_str = ','.join(basic_bookmakers)

#########################################################

In [16]:
################## util functions #######################
#########################################################

#########################################################
# get date as ISO with offset ###########################
def get_ISO_date_with_offset(date, offset):
    # Parse the ISO date string to a datetime object
    date_time_obj = datetime.fromisoformat(date.replace('Z', '+00:00'))

    new_date_time_obj = None
    if (offset > 0):  
        new_date_time_obj = date_time_obj + timedelta(minutes=abs(offset))
    else:
        new_date_time_obj = date_time_obj - timedelta(minutes=abs(offset))

    # Convert the datetime object back to an ISO date string
    new_iso_date_str = new_date_time_obj.isoformat()
    
    # Ensure 'Z' is added for UTC time if the timezone info is UTC
    if new_iso_date_str.endswith('+00:00'):
        new_iso_date_str = new_iso_date_str[:-6] + 'Z'
    
    return new_iso_date_str
#########################################################

#########################################################
# get nba api date as iso ###########################
def get_nba_api_date_as_iso(date):
    # 2023-10-07T16:00:00.000Z -> 2023-10-07T16:00:00Z
    return date.split('.')[0] + 'Z'
#########################################################

#########################################################
# make list distinct by field ###########################
def make_distinct(list_of_objs, field):
    # Step 2: Use a dictionary to track unique objects by the 'id' field
    unique_objects_dict = {obj[field]: obj for obj in list_of_objs}

    # Step 3: Convert the dictionary back to a list
    unique_objects = list(unique_objects_dict.values())
    
    return unique_objects
#########################################################

#########################################################

In [17]:
################## Mongo Functions ######################
#########################################################

#########################################################
# get id and date by collection and season ##############
def get_nba_id_and_date_by_season(field_name, season):
    pipeline = [
        {
            '$match': {
                'season': season,
            }
        },
        {
            '$project': {
                f"{field_name}": 1,
            } 
        }
    ]
    return list(nba_games_historical_collection.aggregate(pipeline))
#########################################################

#########################################################
# get nba games historical objects by season ############
def get_nba_games_historical_objects_by_season(season):
    return list(nba_games_historical_collection.find({ 'season': season }))
#########################################################

#########################################################
# load nba historical odds by season #####################
def load_nba_historical_odds_by_season(season):
    historical_game_objs = get_nba_games_historical_objects_by_season(season)
    res = get_nba_odds_for_given_games(historical_game_objs)
    nba_odds_historical_collection.insert_many(res)
#########################################################
    
#########################################################

In [18]:
################## API functions ########################
#########################################################

#########################################################
# get historical odds data ##############################
def get_historical_odds_data(event_id, date, region, markets, bookmakers):
    # draftkings,fanduel,betmgm bookmaker examples
    url = f"{root}historical/sports/basketball_nba/events/{event_id}/odds?apiKey={api_key}&date={date}&regions={region}&markets={markets}&oddsFormat=american&bookmakers={bookmakers}"
    response = requests.get(url).json()
    try:
        data = response['data']
        res = { 'bookmakerOdds': data['bookmakers'] }
        res = rename_and_remove_fields(res, '_')
        return res['bookmakerOdds']
    except Exception as e:
        print(f"ERROR no data for url: {url}, response: {response}")
        return [] 
#########################################################

#########################################################
# map nba api event to odds api event ###################
def get_nba_odds_for_given_games(game_objs):
    res = []
    for doc in game_objs:
        url = f"{root}historical/sports/basketball_nba/events?apiKey={api_key}&date={get_nba_api_date_as_iso(doc['dateStart'])}"
        response = requests.get(url).json()
        data_list = response['data']

        # find event with corresponding event
        event_obj = next((obj for obj in data_list if obj['home_team'] == doc['teamsHomeName'] and obj['away_team'] == doc['teamsVisitorsName']), None)

        # construct odds obj
        if (event_obj is not None):
            data = event_obj
            data['nbaApiId'] = doc['_id']
            data['oddsApiId'] = data.pop('id')
            data['dateStart'] = data.pop('commence_time')
            data['season'] = doc['season']
            data = rename_and_remove_fields(data, '_')
            data['_id'] = data['oddsApiId']
            data['bookmakerOdds'] = get_historical_odds_data(
                data['oddsApiId'], 
                data['dateStart'], 
                'us', 
                f"{nba_team_market_str},{nba_player_markets_str}",
                basic_bookmakers_str
            )
            res.append(data)
    return make_distinct(res, '_id')
#########################################################

#########################################################

In [19]:
################## Runner Functions #####################
#########################################################

# load_nba_historical_odds_by_season(2023)

#########################################################

In [20]:
###################### Sandbox ##########################
#########################################################
# id_date_objs = get_nba_id_and_date_by_season('dateStart', 2023)
# for d in id_date_objs:
#     nba_game_id = d['_id']
#     res = nba_odds_historical_collection.find({ 'nbaApiId': nba_game_id })
#     if (len(list(res)) == 0):
#         print(f"{nba_game_id} not found!")

# f"{nba_team_market_str},{nba_player_markets_str}"
get_historical_odds_data('be1ee8db7ba20de87a087e8851f9b2f5', '2023-10-25T23:00:00Z', 'us', f"{nba_team_market_str},{nba_player_markets_str}", basic_bookmakers_str) 

#########################################################

ERROR no data for url: https://api.the-odds-api.com/v4/historical/sports/basketball_nba/events/be1ee8db7ba20de87a087e8851f9b2f5/odds?apiKey=REDACTED_ODDS_API_KEY&date=2023-10-25T23:00:00Z&regions=us&markets=h2h,spreads,totals,outrights,player_points,player_rebounds,player_assists,player_threes,player_blocks,player_steals,player_blocks_steals,player_turnovers,player_points_rebounds_assists,player_points_rebounds,player_points_assists,player_rebounds_assists,player_first_basket,player_double_double,player_triple_double&oddsFormat=american&bookmakers=draftkings,fanduel,betmgm, response: {'message': 'Invalid parameter combination. The given markets cannot be used with this sport.', 'error_code': 'INVALID_MARKET_COMBO', 'details_url': 'https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#invalid-market-combo'}


[]